# Notebook 07 _ Préparation du dashboard Streamlit

## Objectif

Ce notebook prépare les données, les modèles et les résultats nécessaires à la
construction d'une application Streamlit consacrée à la prévision énergétique.

Le dashboard permettra notamment de :

- présenter les principales caractéristiques du projet ;
- consulter les performances finales des modèles ;
- visualiser les valeurs observées et les prédictions ;
- explorer les productions photovoltaïque et éolienne ;
- analyser la demande électrique ;
- mettre à disposition une interface claire pour la démonstration du système.

L'application finale sera développée dans le fichier `app/app.py`.

# 1. Importation des bibliothèques

Cette section importe les bibliothèques nécessaires à la préparation des données
et à la vérification des ressources utilisées par l'application Streamlit.

In [1]:
# 1. IMPORTATION DES BIBLIOTHÈQUES

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Bibliothèques importées.")

Bibliothèques importées.


# 2. Définition des chemins

In [2]:
# 2. DÉFINITION DES CHEMINS

PROJECT_DIR = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
APP_DIR = PROJECT_DIR / "app"

APP_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = PROCESSED_DATA_DIR / "energy_features_1h.csv"
FINAL_RESULTS_FILE = RESULTS_DIR / "final_test_results.csv"
VALIDATION_RESULTS_FILE = RESULTS_DIR / "validation_results.csv"

print("Projet :", PROJECT_DIR)
print("Dataset :", DATA_FILE.exists())
print("Résultats finaux :", FINAL_RESULTS_FILE.exists())
print("Résultats validation :", VALIDATION_RESULTS_FILE.exists())
print("Dossier modèles :", MODELS_DIR.exists())
print("Dossier application :", APP_DIR.exists())

Projet : c:\Users\celes\Desktop\PFE
Dataset : True
Résultats finaux : True
Résultats validation : True
Dossier modèles : True
Dossier application : True


# 3. Vérification des modèles sauvegardés

In [3]:
# 3. VÉRIFICATION DES MODÈLES

model_files = {
    "Régression linéaire — PV": "linear_model_pv.pkl",
    "HistGradientBoosting — PV": "boosting_model_pv.pkl",
    "Régression linéaire — Éolien": "linear_model_wind.pkl",
    "HistGradientBoosting — Éolien": "boosting_model_wind.pkl",
    "Régression linéaire — Demande": "linear_model_demand.pkl",
    "HistGradientBoosting — Demande": "boosting_model_demand.pkl"
}

model_status = []

for model_name, filename in model_files.items():
    model_path = MODELS_DIR / filename

    model_status.append({
        "Modèle": model_name,
        "Fichier": filename,
        "Disponible": model_path.exists()
    })

model_status_df = pd.DataFrame(model_status)

display(model_status_df)

,Modèle,Fichier,Disponible
0,Régression linéaire — PV,linear_model_pv.pkl,True
1,HistGradientBoosting — PV,boosting_model_pv.pkl,True
2,Régression linéaire — Éolien,linear_model_wind.pkl,True
3,HistGradientBoosting — Éolien,boosting_model_wind.pkl,True
4,Régression linéaire — Demande,linear_model_demand.pkl,True
5,HistGradientBoosting — Demande,boosting_model_demand.pkl,True


# 4. Chargement des performances finales

Les métriques finales obtenues sur l'ensemble de test sont chargées afin d'être
affichées dans la page consacrée aux performances des modèles.

In [4]:
# 4. CHARGEMENT DES RÉSULTATS FINAUX

final_results_df = pd.read_csv(
    FINAL_RESULTS_FILE
)

print("Dimensions :", final_results_df.shape)

display(final_results_df)

Dimensions : (3, 5)


,Cible,Modèle retenu,MAE_test,RMSE_test,R2_test
0,Demande électrique,HistGradientBoosting,200.832,271.556,0.997
1,Production photovoltaïque,HistGradientBoosting,269.945,470.215,0.990
2,Production éolienne,Régression linéaire,169.930,264.156,0.952


# 5. Création d'un jeu de données léger pour le dashboard

Le dataset complet utilisé pour la modélisation contient un grand nombre de
caractéristiques et son volume est important.

Afin d'améliorer la rapidité du dashboard, un fichier allégé est créé. Il
contient uniquement les variables nécessaires à l'exploration et aux
visualisations interactives.

In [5]:
# 5. CRÉATION DU DATASET LÉGER

dashboard_columns = [
    "Time",
    "Temperature",
    "Humidity",
    "Wind_speed",
    "GHI",
    "PV_production",
    "Wind_production",
    "Electric_demand",
    "target_PV_production_1h",
    "target_Wind_production_1h",
    "target_Electric_demand_1h"
]

dashboard_df = pd.read_csv(
    DATA_FILE,
    usecols=dashboard_columns,
    parse_dates=["Time"]
)

print("Dimensions :", dashboard_df.shape)
print("Mémoire approximative :")
print(
    round(
        dashboard_df.memory_usage(deep=True).sum() / (1024 ** 2),
        2
    ),
    "Mo"
)

display(dashboard_df.head())

Dimensions : (313620, 11)
Mémoire approximative :
26.32 Mo


,Time,GHI,Wind_speed,Humidity,Temperature,PV_production,Wind_production,Electric_demand,target_PV_production_1h,target_Wind_production_1h,target_Electric_demand_1h
0,2019-01-08 00:00:00,0.000,2.920,77.232,9.240,0,814,21490,0.000,658.000,20600.000
1,2019-01-08 00:05:00,0.000,2.920,77.124,9.260,0,807,21411,0.000,667.000,20514.000
2,2019-01-08 00:10:00,0.000,2.920,77.234,9.240,0,801,21389,0.000,689.000,20423.000
3,2019-01-08 00:15:00,0.000,2.920,77.344,9.220,0,780,21266,0.000,715.000,20425.000
4,2019-01-08 00:20:00,0.000,2.920,77.372,9.220,0,764,21206,0.000,715.000,20396.000


In [6]:
# 6. SAUVEGARDE DU DATASET POUR STREAMLIT

DASHBOARD_DATA_FILE = (
    PROCESSED_DATA_DIR
    / "dashboard_data.csv"
)

dashboard_df.to_csv(
    DASHBOARD_DATA_FILE,
    index=False
)

print("Fichier sauvegardé :", DASHBOARD_DATA_FILE)
print("Fichier existant :", DASHBOARD_DATA_FILE.exists())
print(
    "Taille :",
    round(
        DASHBOARD_DATA_FILE.stat().st_size / (1024 ** 2),
        2
    ),
    "Mo"
)

Fichier sauvegardé : c:\Users\celes\Desktop\PFE\data\processed\dashboard_data.csv
Fichier existant : True
Taille : 27.21 Mo


# Conclusion du Notebook 07

Ce notebook a permis de préparer les ressources nécessaires au développement
de l'application Streamlit.

Les principales étapes réalisées sont :

- vérification des modèles entraînés ;
- chargement des résultats finaux ;
- création d'un jeu de données allégé pour le dashboard ;
- sauvegarde des fichiers nécessaires à l'application.

L'étape suivante consiste à développer le dashboard interactif dans le fichier
`app.py`, qui exploitera directement ces ressources afin de proposer une
interface de visualisation et de démonstration du système de prévision
énergétique.